In [1]:
# src/sh_ssw_methods/methods/u_wind_decel.py
from __future__ import annotations
import numpy as np
import pandas as pd
import xarray as xr

In [4]:
from utils.event_iden_funcs import (
    build_events_df,
)

In [5]:
def detect_u_decel_events(
    u: xr.DataArray,
    *,
    # Algorithm controls
    wdw: int = 10,                 # event window length (days)
    drop_per_day: float | None = None,
    mode: str = "decel",           # "decel" or "accel"
    min_gap_days: int = 20,        # next event must be at least this far after the previous
    ratio_thres: float = 1.2,      # max(|window range| / |window end-start|)
    time_dim: str = "time",
    # Metadata (for build_events_df)
    method: str = "u_wind_decel",
    data_source: str | None = None,
    level_hpa: float | None = None,
    latitude: float | None = None,
    lat_band: str | None = None,
) -> tuple[pd.DataFrame, pd.DatetimeIndex, np.ndarray]:
    """
    Identify deceleration/acceleration events in a zonal-mean wind time series.

    Reimplements the logic from Wu et al. (2022)-style detection:
      - event magnitude = u[t+wdw-1] - u[t] (over a wdw-day window)
      - for decel, require diff <= drop_per_day * wdw
        for accel, require diff >= drop_per_day * wdw
      - ratio = (max(window)-min(window)) / |diff|  <= ratio_thres
      - enforce separation by min_gap_days AND, within that gap window,
        keep only the stronger event (more negative for decel, more positive for accel)

    Returns
    -------
    events_df : pd.DataFrame
        tidy table via build_events_df (with only `date` = onset)
    event_dates : pd.DatetimeIndex
        onset dates
    drop_vals : np.ndarray
        event magnitudes (end - start over the window)
    """

    # make sure input is xr.DataArray
    if not isinstance(u, xr.DataArray):
    raise TypeError(
        "detect_u_decel_events expects an xarray.DataArray. "
        "If you have a Dataset, select a variable first, e.g. ds['u']."
    )
    
    if time_dim not in u.dims:
        raise ValueError(f"Expected `{time_dim}` dimension in input DataArray.")
    if mode not in {"decel", "accel"}:
        raise ValueError("mode must be 'decel' or 'accel'.")

    # Choose a sensible default threshold if not provided (matches your original)
    if drop_per_day is None:
        # If level metadata is present, use it; else default to -2 m/s/day (10 hPa-like)
        if level_hpa in (10, 10.0):
            drop_per_day = -2.0 if mode == "decel" else 2.0
        elif level_hpa in (50, 50.0):
            drop_per_day = -1.0 if mode == "decel" else 1.0
        else:
            drop_per_day = -2.0 if mode == "decel" else 2.0

    # 1) Prepare a clean 1-D time series
    t = pd.to_datetime(u[time_dim].to_index())
    s = pd.Series(u.values, index=t)

    # 2) Compute window start→end diff and window range (vectorized)
    # Align "diff" with window START index
    s_end = s.shift(-(wdw - 1))         # value at t+wdw-1
    diff = s_end - s                    # window difference

    # Rolling range over forward-looking window:
    # We can compute a centered rolling on a reversed series to align at window start,
    # or use expanding on slices. Simpler: compute rolling max/min on forward windows via numpy stride.
    # For clarity (and still efficient), do this with pandas rolling on the *reversed* series, then flip back.
    sr = s[::-1]
    rmax = sr.rolling(window=wdw, min_periods=wdw).max()[::-1]
    rmin = sr.rolling(window=wdw, min_periods=wdw).min()[::-1]
    window_range = rmax - rmin

    # Keep only indices where a full window exists
    valid = ~diff.isna()
    diff = diff[valid]
    window_range = window_range[valid]

    # 3) Apply threshold and ratio filters
    drop_thres_total = drop_per_day * wdw
    if mode == "decel":
        cand_mask = (diff <= drop_thres_total)
        stronger_is = np.argmin   # more negative is stronger
        better = lambda a, b: a < b
    else:
        cand_mask = (diff >= drop_thres_total)
        stronger_is = np.argmax   # more positive is stronger
        better = lambda a, b: a > b

    ratio = (window_range / diff.abs()).replace([np.inf, -np.inf], np.nan)
    ratio_mask = ratio <= ratio_thres
    cand_idx = diff.index[cand_mask & ratio_mask & diff.notna() & ratio.notna()]

    if len(cand_idx) == 0:
        events_df = build_events_df(
            dates=[],
            method=method,
            definition=definition or f"{mode}_wdw{wdw}_ratio{ratio_thres:g}_thr{drop_per_day:g}perday",
            data_source=data_source or "",
            threshold=drop_per_day,
            level_hpa=level_hpa,
            latitude=latitude,
            lat_band=lat_band,
            notes=notes or f"min_gap={min_gap_days}d",
        )
        return events_df, pd.DatetimeIndex([]), np.array([])

    # 4) Enforce min-gap and “stronger-in-window” replacement
    cand = pd.DataFrame({
        "onset": cand_idx,
        "diff": diff.loc[cand_idx].values,            # event magnitude over window
    }).sort_values("onset")

    selected_onsets: list[pd.Timestamp] = []
    selected_diffs: list[float] = []

    # We walk candidates in time; if a candidate is within min_gap_days of the last kept event,
    # we keep only the stronger one (compare magnitudes according to mode).
    for onset, dval in zip(cand["onset"].values, cand["diff"].values):
        if not selected_onsets:
            selected_onsets.append(onset)
            selected_diffs.append(dval)
            continue

        gap_days = (onset - selected_onsets[-1]).days
        if gap_days >= min_gap_days:
            # far enough — accept as new event
            selected_onsets.append(onset)
            selected_diffs.append(dval)
        else:
            # within gap — replace if stronger
            if better(dval, selected_diffs[-1]):
                selected_onsets[-1] = onset
                selected_diffs[-1] = dval
            # else keep existing

    event_dates = pd.DatetimeIndex(selected_onsets)
    drop_vals = np.asarray(selected_diffs)

    # 5) Build unified table
    events_df = build_events_df(
        dates=event_dates,
        method=method,
        definition=definition or f"{mode}_wdw{wdw}_ratio{ratio_thres:g}_thr{drop_per_day:g}perday",
        data_source=data_source or "",
        threshold=drop_per_day,
        level_hpa=level_hpa,
        latitude=latitude,
        lat_band=lat_band,
        notes=(notes or "") + f"; min_gap={min_gap_days}d; window={wdw}d",
    )

    return events_df, event_dates, drop_vals


- test function

In [7]:
import os

In [11]:
os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

'/net/cfc/s2s/rachwu/scripts/sh_ssw_impact/aparc_project/SouthernWarmings_module'

In [12]:
# open sample total column ozone data
u1060_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"u1060s_era5_1959_2023.nc"
xu1060 = xr.open_dataset(u1060_path)

In [15]:
xu1060['u1060S']

<xarray.DataArray 'u1060S' (time: 94964)> Size: 380kB
[94964 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 760kB 1959-01-01 ... 2023-12-31T18:00:00
    lat      float64 8B ...
    plev     float64 8B ...

In [27]:
wdw = 10

In [34]:
u = xu1060['u1060S']

In [35]:
time_dim = "time"

In [36]:
u.values

array([-12.660976 , -13.182125 , -13.12456  , ..., -12.2927885,
       -12.8044405, -12.194786 ], dtype=float32)

In [37]:
# 1) Prepare a clean 1-D time series
t = pd.to_datetime(u[time_dim].to_index())
s = pd.Series(u.values, index=t)

# 2) Compute window start→end diff and window range (vectorized)
# Align "diff" with window START index
s_end = s.shift(-(wdw - 1))         # value at t+wdw-1
diff = s_end - s                    # window difference


In [71]:
diff[:20]

time
1959-01-01 00:00:00   -0.554373
1959-01-01 06:00:00   -0.669040
1959-01-01 12:00:00    0.045829
1959-01-01 18:00:00   -1.507934
1959-01-02 00:00:00   -0.532887
1959-01-02 06:00:00   -0.846895
1959-01-02 12:00:00   -1.364522
1959-01-02 18:00:00   -0.701786
1959-01-03 00:00:00   -0.986928
1959-01-03 06:00:00   -0.410512
1959-01-03 12:00:00   -0.690427
1959-01-03 18:00:00   -0.905260
1959-01-04 00:00:00    0.496498
1959-01-04 06:00:00   -0.926814
1959-01-04 12:00:00    0.172320
1959-01-04 18:00:00    0.642544
1959-01-05 00:00:00   -0.888676
1959-01-05 06:00:00    1.181866
1959-01-05 12:00:00   -1.707214
1959-01-05 18:00:00    1.558916
dtype: float32

In [42]:
sr = s[::-1]
rmax = sr.rolling(window=wdw, min_periods=wdw).max()[::-1]
rmin = sr.rolling(window=wdw, min_periods=wdw).min()[::-1]
window_range = rmax - rmin

In [62]:
sr[:20]

time
2023-12-31 18:00:00   -12.194786
2023-12-31 12:00:00   -12.804440
2023-12-31 06:00:00   -12.292789
2023-12-31 00:00:00   -13.633175
2023-12-30 18:00:00   -12.285131
2023-12-30 12:00:00   -13.829944
2023-12-30 06:00:00   -12.292666
2023-12-30 00:00:00   -13.605230
2023-12-29 18:00:00   -12.201206
2023-12-29 12:00:00   -12.737606
2023-12-29 06:00:00   -11.805660
2023-12-29 00:00:00   -12.358904
2023-12-28 18:00:00   -11.176332
2023-12-28 12:00:00   -11.460774
2023-12-28 06:00:00   -10.299802
2023-12-28 00:00:00   -11.117109
2023-12-27 18:00:00    -9.864721
2023-12-27 12:00:00   -10.626980
2023-12-27 06:00:00   -10.209077
2023-12-27 00:00:00   -10.449168
dtype: float32

In [72]:
sr.rolling(window=wdw, min_periods=wdw).max()[:20]-sr.rolling(window=wdw, min_periods=wdw).min()[:20]

time
2023-12-31 18:00:00         NaN
2023-12-31 12:00:00         NaN
2023-12-31 06:00:00         NaN
2023-12-31 00:00:00         NaN
2023-12-30 18:00:00         NaN
2023-12-30 12:00:00         NaN
2023-12-30 06:00:00         NaN
2023-12-30 00:00:00         NaN
2023-12-29 18:00:00         NaN
2023-12-29 12:00:00    1.635158
2023-12-29 06:00:00    2.024283
2023-12-29 00:00:00    2.024283
2023-12-28 18:00:00    2.653612
2023-12-28 12:00:00    2.653612
2023-12-28 06:00:00    3.530142
2023-12-28 00:00:00    3.305429
2023-12-27 18:00:00    3.740509
2023-12-27 12:00:00    2.872885
2023-12-27 06:00:00    2.872885
2023-12-27 00:00:00    2.494183
dtype: float64

In [60]:
sr.rolling(window=wdw, min_periods=wdw).max()[:20]

time
2023-12-31 18:00:00          NaN
2023-12-31 12:00:00          NaN
2023-12-31 06:00:00          NaN
2023-12-31 00:00:00          NaN
2023-12-30 18:00:00          NaN
2023-12-30 12:00:00          NaN
2023-12-30 06:00:00          NaN
2023-12-30 00:00:00          NaN
2023-12-29 18:00:00          NaN
2023-12-29 12:00:00   -12.194786
2023-12-29 06:00:00   -11.805660
2023-12-29 00:00:00   -11.805660
2023-12-28 18:00:00   -11.176332
2023-12-28 12:00:00   -11.176332
2023-12-28 06:00:00   -10.299802
2023-12-28 00:00:00   -10.299802
2023-12-27 18:00:00    -9.864721
2023-12-27 12:00:00    -9.864721
2023-12-27 06:00:00    -9.864721
2023-12-27 00:00:00    -9.864721
dtype: float64

In [52]:
sr.rolling(window=wdw, min_periods=wdw).max()

time
2023-12-31 18:00:00          NaN
2023-12-31 12:00:00          NaN
2023-12-31 06:00:00          NaN
2023-12-31 00:00:00          NaN
2023-12-30 18:00:00          NaN
                         ...    
1959-01-02 00:00:00   -12.924550
1959-01-01 18:00:00   -12.924550
1959-01-01 12:00:00   -12.924550
1959-01-01 06:00:00   -12.924550
1959-01-01 00:00:00   -12.660976
Length: 94964, dtype: float64

In [46]:
s

time
1959-01-01 00:00:00   -12.660976
1959-01-01 06:00:00   -13.182125
1959-01-01 12:00:00   -13.124560
1959-01-01 18:00:00   -13.175361
1959-01-02 00:00:00   -12.924550
                         ...    
2023-12-30 18:00:00   -12.285131
2023-12-31 00:00:00   -13.633175
2023-12-31 06:00:00   -12.292789
2023-12-31 12:00:00   -12.804440
2023-12-31 18:00:00   -12.194786
Length: 94964, dtype: float32

In [73]:
detect_u_decel_events(xu1060['u1060S'])

AttributeError: 'numpy.timedelta64' object has no attribute 'days'

In [75]:
xu1060['u1060S'].time

<xarray.DataArray 'time' (time: 94964)> Size: 760kB
array(['1959-01-01T00:00:00.000000000', '1959-01-01T06:00:00.000000000',
       '1959-01-01T12:00:00.000000000', ..., '2023-12-31T06:00:00.000000000',
       '2023-12-31T12:00:00.000000000', '2023-12-31T18:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 760kB 1959-01-01 ... 2023-12-31T18:00:00
    lat      float64 8B ...
    plev     float64 8B ...
Attributes:
    standard_name:  time
    axis:           T